In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.ops as ops
from torchvision import transforms
from PIL import Image

import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import matplotlib.patches as patches

import tkinter as tk
from tkinter import filedialog, messagebox
import pygame.camera as camera
import pygame

#               
# 1. Детекция лиц (SSD)
#               

@torch.no_grad()
def predict_image(net, img_source, conf_thresh=0.05, iou_thresh=0.45, device='cuda'):
    net.eval()

    if isinstance(img_source, (str, Path)):
        img = Image.open(img_source).convert('RGB')
    else:
        img = img_source.convert('RGB')

    orig_w, orig_h = img.size
    img_array = np.array(img)

    h, w = img_array.shape[:2]
    scale = min(512 / w, 512 / h)
    nw, nh = int(w * scale), int(h * scale)

    img_resized = Image.fromarray(img_array).resize((nw, nh), Image.BILINEAR)
    img_pad = Image.new('RGB', (512, 512), (114, 114, 114))
    img_pad.paste(img_resized, ((512 - nw) // 2, (512 - nh) // 2))

    x = torch.from_numpy(np.array(img_pad)).permute(2, 0, 1).unsqueeze(0).float() / 255.
    x = x.to(device)

    anchors, cls_preds, bbox_preds = net(x)

    anchors = anchors.to(device)
    cls_preds = cls_preds[0]
    bbox_preds = bbox_preds[0]

    scores = F.softmax(cls_preds, dim=1)[:, 1]

    boxes = decode_boxes(bbox_preds, anchors)
    boxes = boxes.clamp(0, 1)

    keep = scores > conf_thresh
    if keep.sum() == 0:
        return np.array([]), np.array([])

    boxes = boxes[keep]
    scores = scores[keep]

    keep = ops.nms(boxes, scores, iou_thresh)
    boxes = boxes[keep]
    scores = scores[keep]

    boxes = boxes.cpu().numpy()
    scores = scores.cpu().numpy()

    pad_x, pad_y = (512 - nw) / 2, (512 - nh) / 2
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] * 512 - pad_x) / scale
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] * 512 - pad_y) / scale

    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, orig_w)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, orig_h)

    return boxes, scores

def decode_boxes(pred, anchors):
    ax = (anchors[:, 0] + anchors[:, 2]) / 2
    ay = (anchors[:, 1] + anchors[:, 3]) / 2
    aw = anchors[:, 2] - anchors[:, 0]
    ah = anchors[:, 3] - anchors[:, 1]

    gx = pred[:, 0] * aw + ax
    gy = pred[:, 1] * ah + ay
    gw = torch.exp(pred[:, 2]) * aw
    gh = torch.exp(pred[:, 3]) * ah

    return torch.stack([
        gx - gw/2, gy - gh/2,
        gx + gw/2, gy + gh/2
    ], dim=1)

class MobileNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU6(),
            nn.Conv2d(32, 32, 3, 1, 1, groups=32), nn.BatchNorm2d(32), nn.ReLU6(),
            nn.Conv2d(32, 64, 1), nn.BatchNorm2d(64), nn.ReLU6(),

            nn.Conv2d(64, 64, 3, 2, 1, groups=64), nn.BatchNorm2d(64), nn.ReLU6(),
            nn.Conv2d(64, 128, 1), nn.BatchNorm2d(128), nn.ReLU6(),

            nn.Conv2d(128, 128, 3, 2, 1, groups=128), nn.BatchNorm2d(128), nn.ReLU6(),
            nn.Conv2d(128, 256, 1), nn.BatchNorm2d(256), nn.ReLU6(),
        )

    def forward(self, x):
        return self.model(x)

class SSD(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = MobileNet()
        self.num_classes = num_classes

        self.sizes = [0.06, 0.1, 0.15]
        self.ratios = [0.5, 1.0, 1.5]
        self.num_anchors = len(self.sizes) * len(self.ratios)

        self.cls_head = nn.Conv2d(256, self.num_anchors * num_classes, 3, padding=1)
        self.box_head = nn.Conv2d(256, self.num_anchors * 4, 3, padding=1)

    def forward(self, x):
        feat = self.backbone(x)
        B, _, H, W = feat.shape

        cls = self.cls_head(feat)
        box = self.box_head(feat)

        cls = cls.permute(0,2,3,1).reshape(B, -1, self.num_classes)
        box = box.permute(0,2,3,1).reshape(B, -1, 4)

        anchors = self.generate_anchors(H, W, x.device)
        return anchors, cls, box

    def generate_anchors(self, H, W, device):
        ys, xs = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
        xs = (xs + 0.5) / W
        ys = (ys + 0.5) / H

        anchors = []
        for s in self.sizes:
            for r in self.ratios:
                w = s * np.sqrt(r)
                h = s / np.sqrt(r)
                anchors.append(torch.stack([
                    xs - w/2, ys - h/2,
                    xs + w/2, ys + h/2
                ], dim=-1))
        anchors = torch.stack(anchors, dim=-2)
        return anchors.reshape(-1,4).clamp(0,1).to(device)

#               
# 2. Модуль камеры
#               

class Camerad:
    def __init__(self, height: int, width: int, save_path: Path):
        camera.init()
        self.cameras = camera.list_cameras()
        if not self.cameras:
            raise RuntimeError("Камеры не найдены.")
        self.cam = camera.Camera(self.cameras[0], (height, width))
        self.cam.start()
        self.save_path = str(save_path)

    def get_image(self):
        pygame.image.save(self.cam.get_image(), self.save_path)
        return self.save_path

    def stop(self):
        if self.cam:
            self.cam.stop()
            camera.quit()

#               
# 3. Распознавание лиц (FaceNet / ArcFace)
#               

class ResNet50Embed(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        from torchvision.models import resnet50, ResNet50_Weights
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.fc = nn.Linear(2048, d, bias=False)
        self.bn = nn.BatchNorm1d(d)
        self.prelu = nn.PReLU(d)

    def forward(self, x):
        x = self.backbone(x).flatten(1)
        f = self.prelu(self.bn(self.fc(x)))
        fn = F.normalize(f, p=2, dim=1)
        return f, fn

class FaceNet(nn.Module):
    def __init__(self, D=512, q_min=0.2, q_max=1.0):
        super().__init__()
        self.backbone = ResNet50Embed(D)
        self.q_min, self.q_max = q_min, q_max
        self.register_buffer("ema_norm", torch.tensor(1.0))

    def _compute_q(self, f_raw: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            n = f_raw.float().norm(dim=1)
            q = (n / self.ema_norm.clamp_min(1e-6)).clamp(self.q_min, self.q_max)
            return q

class FaceRecognizer:
    def __init__(self, model_path, threshold=0.45, device=None):
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.threshold = threshold
        self.gallery = {}

        self.net = FaceNet().to(self.device)
        self._load_weights(model_path)
        self.net.eval()

        self.transform = transforms.Compose([
            transforms.Resize((112, 112)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def _load_weights(self, path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Файл весов не найден: {path}")

        ckpt = torch.load(path, map_location=self.device)
        state_dict = ckpt['model'] if 'model' in ckpt else ckpt
        state_dict = {k: v for k, v in state_dict.items() if "head" not in k}

        self.net.load_state_dict(state_dict, strict=False)
        print(f"Веса распознавания загружены. Эталонная норма: {self.net.ema_norm.item():.2f}")

    def _get_image_tensor(self, img_source):
        if isinstance(img_source, (str, Path)):
            img = Image.open(img_source).convert('RGB')
        else:
            img = img_source.convert('RGB')
        return self.transform(img).unsqueeze(0).to(self.device)

    def _get_k_prototypes(self, embeddings, k=3):
        n = embeddings.size(0)
        if n <= k: return embeddings

        centrs = embeddings[torch.randperm(n)[:k]]
        for _ in range(10):
            sims = torch.mm(embeddings, centrs.t()) #[n, k]
            labels = sims.argmax(dim=1)
            new_centers = []
            for i in range(k):
                mask = labels == i
                if mask.any():
                    new_centers.append(F.normalize(embeddings[mask].mean(0), dim=0))
                else:
                    new_centers.append(embeddings[torch.randint(0, n, (1,))].squeeze())
            centrs = torch.stack(new_centers)
        return centrs

    @torch.no_grad()
    def register_person(self, image_batches, name, k=3):
        files = []
        for batch in image_batches:
            if batch['name'] == name:
                files.extend(batch['paths'])

        if not files:
            print(f"Нет фото для имени {name}.")
            return

        tensors = torch.cat([self._get_image_tensor(f) for f in files])
        f_raw, f_norm = self.net.backbone(tensors)
        q = self.net._compute_q(f_raw)

        good_mask = q > 0.5
        valid_embs = f_norm[good_mask] if good_mask.any() else f_norm

        self.gallery[name] = self._get_k_prototypes(valid_embs, k=k).cpu()
        print(f"Человек '{name}' добавлен с {len(self.gallery[name])} прототипами.")

    @torch.no_grad()
    def recognize(self, img_source):
        if not self.gallery: return "Unknown", 0.0

        _, probe_emb = self.net.backbone(self._get_image_tensor(img_source))
        probe_emb = probe_emb.cpu() # [1, 512]

        best_name, best_score = "Unknown", -1.0

        for name, proto in self.gallery.items():
            sim = torch.mm(probe_emb, proto.t())
            max_sim = sim.max().item()

            if max_sim > best_score:
                best_score = max_sim
                best_name = name

        if best_score >= self.threshold:
            return best_name, best_score
        return "Unknown", best_score


#               
# 4. GUI Приложение
#               

class FaceDetectionApp:
    def __init__(self, root, net, model_path, image_path, device='cuda'):
        self.device = device
        self.engine = FaceRecognizer(model_path=model_path, threshold=0.45, device=self.device)
        self.net = net
        self.image_path = image_path
        self.camera_save_path = str(image_path)

        self.current_image_path = None
        self.current_image = None
        self.image_batches = []
        self.boxes = None
        self.scores = None

        self.root = root
        self.root.title("Распознавание лиц - SSD Neural Network")
        self.root.geometry("900x750")

        # Инициализация камеры
        try:
            self.camera = Camerad(1920, 1080, self.image_path)
            camera_status = "✓ Камера готова"
            camera_color = "green"
        except Exception as e:
            self.camera = None
            camera_status = f"✗ Камера: {str(e)}"
            camera_color = "red"

        #   Верхняя панель управления  
        self.control_frame = tk.Frame(root)
        self.control_frame.pack(fill=tk.X, padx=10, pady=10)

        self.btn_load = tk.Button(self.control_frame, text="📁 Загрузить фото",
                                  command=self.load_image, bg="#FF9800", fg="white")
        self.btn_load.pack(side=tk.LEFT, padx=5)

        self.btn_camera = tk.Button(self.control_frame, text="📷 С камеры",
                                    command=self.capture_from_camera,
                                    bg="#2196F3", fg="white")
        self.btn_camera.pack(side=tk.LEFT, padx=5)

        self.btn_load_multi = tk.Button(self.control_frame,
                                        text="👤 Регистрация лица",
                                        command=self.load_multiple_images_with_name)
        self.btn_load_multi.pack(side=tk.LEFT, padx=5)

        if not self.camera:
            self.btn_camera.config(state=tk.DISABLED)
            self.btn_load_multi.config(state=tk.DISABLED)

        self.btn_detect = tk.Button(self.control_frame, text="🔍 Найти и Распознать",
                                    command=self.find_faces,
                                    bg="#4CAF50", fg="white", font=('Arial', 10, 'bold'))
        self.btn_detect.pack(side=tk.LEFT, padx=5)
        self.btn_detect.config(state=tk.DISABLED)

        self.lbl_camera_status = tk.Label(self.control_frame, text=camera_status, fg=camera_color)
        self.lbl_camera_status.pack(side=tk.LEFT, padx=10)

        self.lbl_status = tk.Label(self.control_frame, text="Статус: Ожидание...", fg="gray")
        self.lbl_status.pack(side=tk.RIGHT)

        #   Панель настроек  
        self.settings_frame = tk.Frame(root)
        self.settings_frame.pack(fill=tk.X, padx=10, pady=5)

        tk.Label(self.settings_frame, text="Confidence:").pack(side=tk.LEFT, padx=5)
        self.conf_var = tk.DoubleVar(value=0.5)
        self.conf_scale = tk.Scale(self.settings_frame, from_=0.1, to=0.9,
                                   resolution=0.1, orient=tk.HORIZONTAL,
                                   variable=self.conf_var, length=150,
                                   command=self._on_slider_change)
        self.conf_scale.pack(side=tk.LEFT, padx=5)

        tk.Label(self.settings_frame, text="IoU:").pack(side=tk.LEFT, padx=5)
        self.iou_var = tk.DoubleVar(value=0.45)
        self.iou_scale = tk.Scale(self.settings_frame, from_=0.1, to=0.8,
                                  resolution=0.05, orient=tk.HORIZONTAL,
                                  variable=self.iou_var, length=150,
                                  command=self._on_slider_change)
        self.iou_scale.pack(side=tk.LEFT, padx=5)

        #  Область для отображения (Matplotlib) 
        self.plot_frame = tk.Frame(root)
        self.plot_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        self.figure = Figure(figsize=(6, 5), dpi=100)
        self.ax = self.figure.add_subplot(111)
        self.ax.axis('off')

        self.canvas = FigureCanvasTkAgg(self.figure, master=self.plot_frame)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

        self.canvas._tkcanvas.pack(fill=tk.BOTH, expand=True)
        self.canvas.draw_idle()

        self.conf_scale.bind('<ButtonRelease-1>', lambda e: self._on_slider_change(None))
        self.iou_scale.bind('<ButtonRelease-1>', lambda e: self._on_slider_change(None))

        #   Панель результатов  
        self.result_frame = tk.Frame(root)
        self.result_frame.pack(fill=tk.X, padx=10, pady=5)

        self.lbl_result = tk.Label(self.result_frame, text="Лиц найдено: 0",
                                   font=('Arial', 12, 'bold'), fg="blue")
        self.lbl_result.pack(side=tk.LEFT)

        self.btn_save = tk.Button(self.result_frame, text="💾 Сохранить результат",
                                  command=self.save_result, bg="#9C27B0", fg="white")
        self.btn_save.pack(side=tk.RIGHT)
        self.btn_save.config(state=tk.DISABLED)

    def _on_slider_change(self, value):
        """Вызывается при движении любого ползунка"""
        # Обновляем статус
        self.lbl_status.config(
            text=f"Conf: {self.conf_var.get():.2f} | IoU: {self.iou_var.get():.2f}", 
            fg="blue"
        )
        
        # Перерисовываем если есть детекции
        if self.boxes is not None and self.current_image is not None:
            self.display_image(self.current_image, self.boxes, self.scores)

    def load_image(self):
        file_path = filedialog.askopenfilename(
            filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp")]
        )
        if file_path:
            try:
                img = Image.open(file_path).convert('RGB')
                self.current_image = img
                self.current_image_path = Path(file_path)

                self.ax.clear()
                self.ax.imshow(img)
                self.ax.axis('off')
                self.canvas.draw()

                self.btn_detect.config(state=tk.NORMAL)
                self.lbl_status.config(text="Статус: Фото загружено", fg="green")
                self.lbl_result.config(text="Лиц найдено: 0")
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось открыть файл:\n{e}")

    def capture_from_camera(self):
        if not self.camera:
            return
        try:
            self.lbl_status.config(text="Статус: Захват с камеры...", fg="orange")
            self.root.update()

            self.camera.get_image()
            img = Image.open(self.camera_save_path).convert('RGB')
            self.current_image_path = Path(self.camera_save_path)
            self.current_image = img

            self.ax.clear()
            self.ax.imshow(img)
            self.ax.axis('off')
            self.canvas.draw()

            self.btn_detect.config(state=tk.NORMAL)
            self.lbl_status.config(text="Статус: Снимок готов", fg="green")
            self.lbl_result.config(text="Лиц найдено: 0")
        except Exception as e:
            messagebox.showerror("Ошибка", f"Не удалось захватить изображение:\n{e}")
            self.lbl_status.config(text="Статус: Ошибка камеры", fg="red")

    def find_faces(self):
        if self.current_image is None:
            return
        try:
            self.lbl_status.config(text="Статус: Обработка нейросетью...", fg="orange")
            self.btn_detect.config(state=tk.DISABLED)
            self.root.update()

            conf_thresh = self.conf_var.get()
            iou_thresh = self.iou_var.get()

            self.boxes, self.scores = predict_image(
                self.net, self.current_image,
                conf_thresh=conf_thresh, iou_thresh=iou_thresh, device=self.device
            )

            self.display_image(self.current_image, self.boxes, self.scores)

            num_faces = len(self.boxes) if self.boxes is not None else 0
            self.lbl_result.config(text=f"Лиц найдено: {num_faces}")
            self.lbl_status.config(text=f"Статус: Найдено {num_faces} лиц", fg="green" if num_faces > 0 else "gray")

            self.btn_save.config(state=tk.NORMAL)
            self.btn_detect.config(state=tk.NORMAL)
        except Exception as e:
            messagebox.showerror("Ошибка детекции", f"Ошибка при обработке:\n{e}")
            self.lbl_status.config(text="Статус: Ошибка", fg="red")
            self.btn_detect.config(state=tk.NORMAL)

    def display_image(self, img, boxes=None, scores=None):
        self.ax.clear()
        self.ax.imshow(img)

        if boxes is not None and scores is not None and len(boxes) > 0:
            conf_thresh = self.conf_var.get()
            for box, score in zip(boxes, scores):
                if score > conf_thresh:
                    x1, y1, x2, y2 = box.astype(int)

                    # Защита от кривых рамок
                    if x2 - x1 < 10 or y2 - y1 < 10:
                        continue

                    # Вырезаем лицо в памяти (без сохранения на диск)
                    face_crop = img.crop((x1, y1, x2, y2))

                    # Передаем обрезанное лицо напрямую в модель распознавания
                    pers_name, conf = self.engine.recognize(face_crop)

                    # Отрисовка
                    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                             linewidth=3, edgecolor='red', facecolor='none')
                    self.ax.add_patch(rect)

                    self.ax.text(x1, y1-10, f'{pers_name}: {score:.3f}',
                                 color='red', fontsize=11, fontweight='bold',
                                 bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))

        self.ax.axis('off')
        self.figure.tight_layout()
        self.canvas.draw()

    def load_multiple_images_with_name(self):
        """Регистрация нового человека с автоматической обрезкой лица (Smart Crop)"""
        dialog = tk.Toplevel(self.root)
        dialog.title("Создание группы изображений")
        dialog.geometry("450x250")
        dialog.resizable(False, False)
        dialog.transient(self.root)
        dialog.grab_set()

        tk.Label(dialog, text="Введите имя человека:").pack(pady=10)
        entry_name = tk.Entry(dialog, width=40)
        entry_name.pack(pady=5)
        entry_name.focus_set()

        tk.Label(dialog, text="Количество фотографий (ракурсов):").pack(pady=5)
        spin_count = tk.Spinbox(dialog, from_=1, to=20, width=5, font=("Arial", 12))
        spin_count.pack(pady=5)
        spin_count.delete(0, tk.END)
        spin_count.insert(0, "5")

        def on_confirm():
            name = entry_name.get().strip()
            if not name:
                messagebox.showwarning("Внимание", "Введите имя!")
                return
            try:
                count = int(spin_count.get())
            except ValueError:
                messagebox.showwarning("Внимание", "Некорректное число!")
                return

            captured_paths = []
            dialog.destroy() # Закрываем окно, чтобы не мешало

            self.lbl_status.config(text=f"Статус: Подготовка к съемке '{name}'...", fg="orange")
            self.root.update()

            for i in range(count):
                time.sleep(0.5) # Даем время сменить позу
                try:
                    # 1. Захват кадра
                    surface = self.camera.cam.get_image()
                    raw_str = pygame.image.tostring(surface, "RGB", False)
                    full_img = Image.frombytes("RGB", surface.get_size(), raw_str)

                    # 2. Поиск лица через SSD
                    boxes, scores = predict_image(self.net, full_img, conf_thresh=0.5, iou_thresh=0.45, device=self.device)

                    # 3. Умная обрезка (Берем самое уверенное лицо)
                    if len(boxes) > 0:
                        best_idx = np.argmax(scores)
                        x1, y1, x2, y2 = boxes[best_idx].astype(int)

                        if x2 - x1 > 20 and y2 - y1 > 20: # Проверка на размер
                            face_crop = full_img.crop((x1, y1, x2, y2))
                            # Сохраняем ТОЛЬКО лицо
                            image_path = Path.cwd().parent / f"{name}_{i}.jpg"
                            face_crop.save(image_path)
                            captured_paths.append(image_path)
                            self.lbl_status.config(text=f"Захвачено лиц: {len(captured_paths)}/{count}", fg="blue")
                            self.root.update()
                except Exception as e:
                    print(f"Ошибка кадра {i}: {e}")

            if captured_paths:
                self.image_batches.append({'name': name, 'paths': captured_paths})
                # Регистрация
                self.engine.register_person(self.image_batches, name)
                self.lbl_status.config(text=f"Статус: '{name}' успешно зарегистрирован!", fg="green")
            else:
                messagebox.showwarning("Провал", "Не удалось найти лица на кадрах. Попробуйте еще раз с лучшим освещением.")

        btn_frame = tk.Frame(dialog)
        btn_frame.pack(pady=15)
        tk.Button(btn_frame, text="OK", command=on_confirm, width=10, bg="#4CAF50", fg="white").pack(side=tk.LEFT, padx=10)
        tk.Button(btn_frame, text="Отмена", command=dialog.destroy, width=10).pack(side=tk.LEFT, padx=10)

    def save_result(self):
        if self.current_image is None: return
        file_path = filedialog.asksaveasfilename(defaultextension=".png", filetypes=[("PNG files", "*.png")])
        if file_path:
            try:
                fig, ax = plt.subplots(1, figsize=(10, 10))
                ax.imshow(self.current_image)
                if self.boxes is not None and self.scores is not None:
                    conf_thresh = self.conf_var.get()
                    for box, score in zip(self.boxes, self.scores):
                        if score > conf_thresh:
                            x1, y1, x2, y2 = box
                            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=3, edgecolor='red', facecolor='none')
                            ax.add_patch(rect)
                            img = self.current_image
                            face_crop = img.crop((x1, y1, x2, y2))
                            pers_name, conf = self.engine.recognize(face_crop)
                            ax.text(x1, y1-10, f'{pers_name}: {score:.3f}', color='red', fontsize=12, fontweight='bold')
                ax.axis('off')
                plt.savefig(file_path, dpi=150, bbox_inches='tight')
                plt.close(fig)
                messagebox.showinfo("Успех", f"Результат сохранён:\n{file_path}")
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось сохранить:\n{e}")

    def on_closing(self):
        if self.camera:
            self.camera.stop()
        self.root.destroy()
        exit(0)

#               
# Запуск
#               
if __name__ == "__main__":
    current_dir = Path.cwd()
    parent_dir = current_dir.parent
    parent_dir.mkdir(parents=True, exist_ok=True)

    image_path = parent_dir / 'image.jpg'
    best_path_ssd = parent_dir / 'best_ssd.pth'
    MODEL_PATH = parent_dir / 'best.pt'

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Используемое устройство: {device}")

    if not best_path_ssd.is_file():
        raise RuntimeError(f"Файл best_ssd.pth не найден по пути: {best_path_ssd}")
    if not MODEL_PATH.is_file():
        raise RuntimeError(f"Файл best.pt не найден по пути: {MODEL_PATH}")

    print("Загрузка моделей...")
    net = SSD(num_classes=2).to(device)
    net.load_state_dict(torch.load(best_path_ssd, map_location=device))
    net.eval()
    print("Модели загружены успешно!")

    root = tk.Tk()
    app = FaceDetectionApp(root, net, model_path=MODEL_PATH, image_path=image_path, device=device)
    root.protocol("WM_DELETE_WINDOW", app.on_closing)

    print("\n  Приложение запущено  ")
    root.mainloop()


c:\Users\Kolya\AppData\Local\Programs\Python\Python312\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
Используемое устройство: cuda
Загрузка моделей...
Модели загружены успешно!


C:\Users\Kolya\AppData\Local\Temp\ipykernel_14080\65124258.py:678: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load(best_path_ssd, map_location=d

Веса распознавания загружены. Эталонная норма: 0.27

  Приложение запущено  


: 